In [ ]:
!python -m pip install prophet


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from prophet import Prophet

from config.settings import (
    SHEET_ID_ALL_MEDIA, SHEET_ID_PRESUPUESTO_OPTIMIZADO,
    COLUMNAS_NUMERICAS, FECHA_INICIO, FECHA_FIN,
    CHANGEPOINT_PRIOR_SCALE, FORECAST_HORIZON_DAYS, HOLIDAYS,
)
from src.campaigns import CAMPANAS_INNOVA, CAMPANAS_FESA
from src.metrics import (
    analizar_campanas, promedio_campana,
    analizar_roas_mensual, boxplot_campana, histograma_campana,
)
from src.continuity import (
    analizar_continuidad_campana_revenue,
    analizar_continuidad_campana_gasto,
)
from src.prophet_model import (
    get_revenue_por_campana, get_spend_por_campana,
    construir_holidays, entrenar_prophet, evaluar_modelo,
)

In [ ]:
# # # # Conectar a Drive
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

# Análisis histórico por campaña


In [ ]:
#all media
worksheet = gc.open_by_key("114qrqgEosT7FSLDpA210ksnuBHVjUUlpdfKArm-6Nq8").sheet1
rows = worksheet.get_all_values()
df_all_media = pd.DataFrame.from_records(rows)
df_all_media = pd.DataFrame.from_records(rows[1:], columns=rows[0])



In [ ]:
# Lista de columnas numéricas definida en config/settings.py
todos_flotan = COLUMNAS_NUMERICAS

In [ ]:
array_fesa = ["Clicks", "Impresiones", "Spend"]

In [ ]:
# Convierte todas las columnas numéricas de una vez
df_all_media[todos_flotan] = df_all_media[todos_flotan].astype(float, errors='ignore')

In [ ]:
# Campañas de Innova — lista definida en src/campaigns.py
campanas = CAMPANAS_INNOVA

In [ ]:
# Campañas de Fesa — lista definida en src/campaigns.py
campanas = CAMPANAS_FESA

In [ ]:
#ya están agrupadas, más bien quiero un filtro
#necesito comparar la fila del df vs un array en la columna df[campaign.name]. Si la campaña está filtrala.
boolean_mask = df_all_media['campaign.name'].isin(campanas)


In [ ]:
filtered_df = df_all_media[boolean_mask]
print(filtered_df)

# Comparativa optimización de objetivos MMM vs distribución presupuestaria actual


In [ ]:
df_presupuesto = df_all_media.groupby("Objetivo por medio")['Spend'].agg(['sum'])

In [ ]:
df_presupuesto['presupuesto en porcentaje actual'] = (
    df_presupuesto['sum'] / df_presupuesto['sum'].sum() * 100
)

In [ ]:
df_presupuesto_actual = df_presupuesto.sort_values('presupuesto en porcentaje actual', ascending=False)

In [ ]:
#pasa sum a números normales
df_presupuesto_actual['sum'] = df_presupuesto_actual['sum'].astype(float)

In [ ]:
df_presupuesto_actual.head(25)

In [ ]:
worksheet = gc.open_by_key("1WvlbVZJWePjZQZ5g3eszzLiqmUAfrJmu-b3ayu8n1vo").sheet1
rows = worksheet.get_all_values()
presupuesto_optimizado = pd.DataFrame.from_records(rows)
presupuesto_optimizado = pd.DataFrame.from_records(rows[1:], columns=rows[0])


In [ ]:
presupuesto_optimizado['presupuesto recomendado por modelo'] = presupuesto_optimizado['presupuesto recomendado por modelo'].astype(float)

In [ ]:
presupuesto_optimizado.head()

In [ ]:
presupuestos_unidos = pd.merge(
    df_presupuesto_actual,
    presupuesto_optimizado,
    on = 'Objetivo',
    how = 'left'
)

# Presupuesto actual vs presupuesto recomendado por el modelo


In [ ]:
#podemos ver que la recomendación de aumentar a PMAX ya no va pues ya la ejecutaron. Ya se pasaron en ASC y PLA. PLA no le puedo bajar porque ahorita mismo hay promociones
#los accionables de un MMM es usarlo para reasignar presupuesto en un objetivo de medios particular. Creo que podemos proponerles lo siguiente; el MMM sirve para asignar presupuestos en un acuerto
#como mínima cantidad de tiempo medible.
#entonces lo único que se me ocurre es; el cambio no es de aquí a una semana, si no de aquí a un cuarto.
#es que no hay una compaginación entre el mROAs y el ROAS de atribución que están haciendo.
#Tiene sentido que se le baje porque ASC es como el PMAX de facebook. A PLA también se le sugiere bajar pero creo que la tendencia del objetivo es lo que importa.
#El media mix no nos dice a, tiene bajo ROAS cierto objetivo, nos dice que ya no tiene tanto espacio de crecimiento. Por ejemplo PMAx ya llegó a donde debía.
#Brand terms no lo podemos tocar, reach no lo modelé. Traffic carece de sentido. Facebook conversion ya llegó a donde debía llegar. Search no lo medí
#Básicamente hay 5 candidatos. ASC quitarle. PLA quitarle mucho. DPA aumentar el doble. DABA aumentar el doble. DPA me dijeron que no tenía sentido porque remarketing ya estaba topado
#queda aumentarle a DABA. ¿en qué campañas? ¿crear alguna nueva?
#bajarle a PLA. Lo ideal sería a total objetivo pero weno.
#aumentarle a DABA nos da proospección de nuevos clientes que pueden alimentar DPA.
#entonces podemos empezar de a poco. Una campaña de PLA a la que le baje lo que le voy a subir a alguna de DABA.
#haz el análisis para la peor campaña de PLA que siga existiendo


presupuestos_unidos.plot(
    x='Objetivo',
    y=['presupuesto recomendado por modelo', 'presupuesto en porcentaje actual'],
    kind='bar'
)

# Desempeño campaña por métrica y rango temporal


In [ ]:
# analizar_campanas importada desde src/metrics.py

# Ranking general de campañas

In [ ]:
#creo que con esta puedo rankear a las campañas. PLA es la que mejor le fue de las normales
resultado = analizar_campanas(
    df_all_media,
    fecha_inicio='2025-01-01',
    fecha_fin='2026-03-10',
    metrica='ROAS'
)

resultado.head(28)

In [ ]:
#Ranking por objetivo de campañas
resultado[resultado["campaign.name"].str.contains("_daba_", na=False)]

In [ ]:
# analizar_continuidad_campana_revenue importada desde src/continuity.py

# Serie temporal en el tiempo


In [ ]:
resultado = analizar_continuidad_campana_revenue(
    df_all_media,
    campaign_name="fb_innova_ecomm_think_auct_daba_compra_always-on",
    fecha_inicio="2025-01-01",
    fecha_fin="2026-03-09"
)

resultado

# Promedio histórico

In [ ]:
# promedio_campana importada desde src/metrics.py

In [ ]:
promedio_campana(df_all_media, "aw_innova_ecomm_do_pla_cat_ao_villanos_hp_nobt_v2", "2026-01-01", "2026-03-09")

In [ ]:
# analizar_roas_mensual importada desde src/metrics.py

In [ ]:
#creo que esta es la más fácil de evaluar y me permite observar si históricamente la campaña ha ido creciendo
roas_analizable = analizar_roas_mensual(
    df_all_media,
    campaign_name="fb_innova_ecomm_think_auct_daba_compra_always-on",
    fecha_inicio="2025-01-01",
    fecha_fin="2026-03-09"
)

In [ ]:
promedio_campana(
    df_all_media,
    'aw_innova_ecomm_do_pla_cat_ao_calzado_v2',
    '2026-02-22',
    '2026-03-09'
)

# Boxplot

In [ ]:
# boxplot_campana importada desde src/metrics.py

In [ ]:
outliers = boxplot_campana(
    df_all_media,
    campaign_name="fb_innova_ecomm_think_auct_daba_compra_always-on",
    metric="Revenue con IVA"
)

# Histograma

In [ ]:
# histograma_campana importada desde src/metrics.py

In [ ]:
histograma_campana(df_all_media, "fb_innova_ecomm_think_auct_daba_compra_always-on", "Revenue con IVA")

# Proporción de la inversión

# tomar datos de inversión de la campaña


In [ ]:
# get_revenue_por_campana importada desde src/prophet_model.py

In [ ]:
# get_spend_por_campana importada desde src/prophet_model.py

# Campaña a poner en prophet

In [ ]:

df_prophet = get_revenue_por_campaña(
    "aw_innova_ecomm_do_pla_cat_ao_villanos_hp_nobt_v2",
    df_all_media
)



In [ ]:
df_gasto = get_spend_por_campaña("aw_innova_ecomm_do_pla_cat_ao_villanos_hp_nobt_v2",
    df_all_media
)

In [ ]:
#cambio nombre columna
df_prophet.columns = ['ds', 'y']


# Método más tranquilo

In [ ]:
df_prophet = get_revenue_por_campana(
    "aw_innova_ecomm_do_pla_cat_ao_villanos_hp_nobt_v2", df_all_media
)

holidays = construir_holidays(HOLIDAYS)
model, forecast = entrenar_prophet(
    df_prophet,
    changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
    holidays=holidays,
    periodos=FORECAST_HORIZON_DAYS,
)
fig = model.plot(forecast)

# Forecast con eventos especiales y trend flexible

In [ ]:
# Los holidays ahora se definen en config/settings.py y se construyen
# con construir_holidays(HOLIDAYS) — ver celda anterior.

In [ ]:
df_prophet.head()

#Entrenamiento del modelo

In [ ]:
model, forecast = entrenar_prophet(
    df_prophet,
    changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
    holidays=holidays,
    periodos=FORECAST_HORIZON_DAYS,
)
fig = model.plot(forecast)

In [ ]:
forecast.head()

In [ ]:
fig = m.plot_components(forecast)
#el trend nos dice que el revenue de la campaña va a la baja desde enero y que lo más probable es que allí se quede
#entonces toca ir armando el dashboard; primero poner las campañas que vamos a bajar


# Cross Validation


In [ ]:
df_cv, df_perf = evaluar_modelo(model, initial='200 days', period='7 days', horizon='7 days')

In [ ]:
df_cv.head()


In [ ]:
df_perf.head()

In [ ]:
from prophet.diagnostics import register_performance_metric, rolling_mean_by_h
import numpy as np
@register_performance_metric
def mase(df, w):
    """Mean absolute scale error

        Parameters
        ----------
        df: Cross-validation results dataframe.
        w: Aggregation window size.

        Returns
        -------
        Dataframe with columns horizon and mase.
    """
    e = (df['y'] - df['yhat'])
    d = np.abs(np.diff(df['y'])).sum()/(df['y'].shape[0]-1)
    se = np.abs(e/d)
    if w < 0:
        return pd.DataFrame({'horizon': df['horizon'], 'mase': se})
    return rolling_mean_by_h(
        x=se.values, h=df['horizon'].values, w=w, name='mase'
    )

df_mase = performance_metrics(df_cv, metrics=['mase'])
df_mase.head()


# Gráfica del error

In [ ]:
# Python
from prophet.plot import plot_cross_validation_metric
fig = plot_cross_validation_metric(df_cv, metric='mase')

In [ ]:
df_cv

# Aplicación de los insights

In [ ]:
# get_spend_por_campaña ya está definida en la celda 47, no se redefine aquí.
# La función original devuelve el spend diario agrupado por campaña.

In [ ]:
#ya tenemos un revenue predicho. Toca ver cuál es
revenue_predicho = forecast[['ds','yhat']].tail(7)

In [ ]:
revenue_predicho.head()

In [ ]:
ultimos_dias = revenue_predicho.head(7)
revenue_total = ultimos_dias["yhat"].sum()

In [ ]:
revenue_total

In [ ]:
# analizar_continuidad_campana_gasto importada desde src/continuity.py

In [ ]:
gasto = analizar_continuidad_campana_gasto(
    df_all_media,
    campaign_name="aw_innova_ecomm_do_pla_cat_ao_villanos_hp_nobt_v2",
    fecha_inicio="2025-12-01",
    fecha_fin="2026-02-28"
)

gasto